# Running the Models with Various Preprocessing Techniques and K-Fold Cross-Validation

## Setup

In [1]:
# for importing utils from python scripts in parent directories

import sys
sys.path.append('../utils')
sys.path.append('..')

In [2]:
# suppress TensorFlow warnings and logs

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all, 1 = info, 2 = warning, 3 = error

import warnings
warnings.filterwarnings('ignore')

# If you use logging, also suppress it:
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [3]:
import tensorflow as tf


gpus = tf.config.experimental.list_physical_devices('GPU')
print("GPUs available:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Defining the image processing function

In [4]:
import numpy as np
import cv2

def preprocess_images(df, preproc_fn):
    X = []
    y = []
    for idx, row in df.iterrows():
        img_path = row['image_path']
        label = row['label']
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
        processed = preproc_fn(img)
        X.append(processed)
        y.append(label)
    X = np.stack(X)
    y = np.stack(y)
    return X, y

## Get the training and testing data

In [5]:
# if not using folds during training, load the train and test sets directly

import pandas as pd

train_df = pd.read_csv('../data/train_split.csv')
test_df = pd.read_csv('../data/test_split.csv')

print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Train set: 1230 samples
Test set: 410 samples


In [6]:
label_mapping = {
    '1': 0,
    '2': 1,
    '3': 2,
    '4a': 3,
    '4b': 4,
    '4c': 5,
    '5': 6,
    '6': 7
}

train_df['label'] = train_df['label'].map(label_mapping)
test_df['label'] = test_df['label'].map(label_mapping)

print('Label mapping applied. Unique train labels:', train_df['label'].unique())
print('Label mapping applied. Unique test labels:', test_df['label'].unique())
print(train_df['label'].isnull().sum(), test_df['label'].isnull().sum())
print(train_df['label'].unique(), test_df['label'].unique())

Label mapping applied. Unique train labels: [6 1 0 5 2 7 3 4]
Label mapping applied. Unique test labels: [1 6 0 5 2 4 3 7]
0 0
[6 1 0 5 2 7 3 4] [1 6 0 5 2 4 3 7]


## Running the Models

### Model Running Function

In [7]:
import numpy as np
from keras.utils import to_categorical
from keras import backend as K

def run_model_with_preprocessing(
    train_df, test_df, preproc_fn, model_name, model_fn, num_classes=8, batch_size=8, epochs=15
):
    X_train, y_train = preprocess_images(train_df, preproc_fn)
    X_test, y_test = preprocess_images(test_df, preproc_fn)
    X_train = np.expand_dims(X_train, -1)
    X_test = np.expand_dims(X_test, -1)
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0

    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)

    if model_name.lower() in ['custom cnn']:
        input_shape = X_train.shape[1:]
        X_tr, X_te = X_train, X_test
    else:
        X_train_3ch = np.repeat(X_train, 3, axis=-1)
        X_test_3ch = np.repeat(X_test, 3, axis=-1)
        input_shape = X_train_3ch.shape[1:]
        X_tr, X_te = X_train_3ch, X_test_3ch
        batch_size = 4

    model = model_fn(input_shape=input_shape, num_classes=num_classes, loss='categorical_crossentropy')
    history = model.fit(
        X_tr, y_train,
        validation_data=(X_te, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2
    )
    val_accuracies = history.history['val_accuracy']
    best_epoch = int(np.argmax(val_accuracies)) + 1
    best_val_acc = float(np.max(val_accuracies))

    K.clear_session()
    del model
    import gc; gc.collect()

    return best_val_acc, best_epoch, history.history

### Auxiliary Functions

In [8]:
def merge_train_test(train_df, test_df):
    train_df['set'] = 'train'
    test_df['set'] = 'test'
    merged_df = pd.concat([train_df, test_df], ignore_index=True)
    return merged_df

In [9]:
import os

def check_if_model_exists(preproc_id, model_name, combo_str):
    history_file = f"../data/results/history_{preproc_id}_{model_name}_{combo_str.replace(', ', '_').replace('=', '-')}.csv"
    return os.path.exists(history_file)

### Running the models

In [ ]:
import itertools
from utils.preprocessing import preprocessing_methods
from utils.models import MODEL_BUILDERS
import time
from sklearn.model_selection import StratifiedKFold
import pandas as pd

fold = 4 # set to the amount of folds used (0 for no folds)
batch_size = 8
run_skip = False # if True, will skip already ran models

for preproc_id, preproc_info in preprocessing_methods.items():
    param_names = list(preproc_info['params'].keys())
    param_values = [preproc_info['params'][k] for k in param_names]
    for param_combo in itertools.product(*param_values):
        param_dict = dict(zip(param_names, param_combo))
        def preproc_fn(img, func=preproc_info['func'], params=param_dict):
            return func(img, **params)
        combo_str = ', '.join([f"{k}={v}" for k, v in param_dict.items()])
        print(f"\n=== Preprocessing: {preproc_id} ({combo_str}) ===")
        for model_name, model_fn in MODEL_BUILDERS.items():
            print(f"--> Current Model: {model_name} <--")
            if run_skip and check_if_model_exists(preproc_id, model_name, combo_str):
                print(f"Skipping {preproc_id} [{model_name} - {combo_str}] as it has already been run.")
                continue
            model_time = time.time()

            if fold == 0:
                # use train_df and test_df as loaded, do NOT merge
                train_set, test_set = train_df, test_df

                best_val_acc, best_epoch, history_dict = run_model_with_preprocessing(
                    train_set, test_set, preproc_fn, model_name, model_fn, num_classes=8, batch_size=batch_size, epochs=15
                )
                print(f"Best val accuracy for {preproc_id} [{model_name} - {combo_str}]: {best_val_acc:.4f} at epoch {best_epoch}")

            else:
                # if using folds, only get the best fold data
                merged_df = merge_train_test(train_df, test_df)
                X = merged_df['image_path']
                y = merged_df['label']
                skf = StratifiedKFold(n_splits=fold, shuffle=True, random_state=42)
                fold_results = []
                for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                    print(f"Running fold {fold_idx+1}/{fold}...")
                    train_set = merged_df.iloc[train_idx].reset_index(drop=True)
                    test_set = merged_df.iloc[test_idx].reset_index(drop=True)
                    best_val_acc, best_epoch, history_dict = run_model_with_preprocessing(
                        train_set, test_set, preproc_fn, model_name, model_fn, num_classes=8, batch_size=batch_size, epochs=15
                    )
                    fold_results.append({
                        'fold': fold_idx+1,
                        'best_val_acc': best_val_acc,
                        'best_epoch': best_epoch
                    })
                    print(f"Fold {fold_idx+1}: Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

                # get the best fold (highest val accuracy)
                best_fold = max(fold_results, key=lambda x: x['best_val_acc'])
                print(f"Best fold for {preproc_id} [{model_name} - {combo_str}]: Fold {best_fold['fold']} with val accuracy {best_fold['best_val_acc']:.4f} at epoch {best_fold['best_epoch']}")
                
                # save best fold results to CSV for run_skip
                history_file = f"../data/results/history_{preproc_id}_{model_name}_{combo_str.replace(', ', '_').replace('=', '-')}.csv"
                pd.DataFrame([best_fold]).to_csv(history_file, index=False)

            end_model_time = time.time()
            elapsed = end_model_time - model_time
            h = int(elapsed // 3600)
            m = int((elapsed % 3600) // 60)
            s = int(elapsed % 60)
            parts = []
            if h > 0:
                parts.append(f"{h}h")
            if m > 0:
                parts.append(f"{m}min")
            if s > 0 or not parts:
                parts.append(f"{s}sec")
            print(f"Time taken for {preproc_id} [{model_name}]: {' '.join(parts)}\n")


=== Preprocessing: denoise (kernel_size=(3, 3), sigma=0) ===
--> Current Model: custom cnn <--
Running fold 1/4...
Epoch 1/15
154/154 - 29s - loss: 1.6283 - accuracy: 0.4772 - val_loss: 1.8408 - val_accuracy: 0.5366 - 29s/epoch - 187ms/step
Epoch 2/15
154/154 - 15s - loss: 1.5203 - accuracy: 0.5211 - val_loss: 1.6869 - val_accuracy: 0.5366 - 15s/epoch - 95ms/step
Epoch 3/15
154/154 - 15s - loss: 1.4859 - accuracy: 0.5276 - val_loss: 1.5154 - val_accuracy: 0.5366 - 15s/epoch - 94ms/step
Epoch 4/15
154/154 - 15s - loss: 1.4602 - accuracy: 0.5293 - val_loss: 1.4860 - val_accuracy: 0.5366 - 15s/epoch - 95ms/step
Epoch 5/15
154/154 - 15s - loss: 1.4732 - accuracy: 0.5301 - val_loss: 1.4610 - val_accuracy: 0.5366 - 15s/epoch - 95ms/step
Epoch 6/15
154/154 - 15s - loss: 1.4569 - accuracy: 0.5276 - val_loss: 1.4688 - val_accuracy: 0.5366 - 15s/epoch - 95ms/step
Epoch 7/15
154/154 - 15s - loss: 1.4529 - accuracy: 0.5309 - val_loss: 1.4334 - val_accuracy: 0.5366 - 15s/epoch - 95ms/step
Epoch 8/